# Debugging `get_standings_table`

This notebook sets up the environment to debug the `get_standings_table` function from `data.espn_mlb_utilities`. 
The issue states: the league started with week 2, but the standing table has data for the first week, but only for 6 teams, creating an imbalance.

In [1]:
import sys
import os
import pandas as pd
import json
import logging

# Add the project root to the path so we can import from the project
sys.path.append(os.path.abspath('.'))

from data.espn_mlb_utilities import get_standings_table, get_category_stats, load_view_json
from utils.config import DEFAULT_LEAGUE_ID, DEFAULT_YEAR, DEFAULT_SCORING_PERIOD_WEEK

logging.basicConfig(level=logging.INFO)

2026-05-03 09:10:36.791 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-03 09:10:36.793 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [2]:
# Setup parameters
league_id = DEFAULT_LEAGUE_ID
year = DEFAULT_YEAR
scoring_period_id = 5 #DEFAULT_SCORING_PERIOD_WEEK # Adjust this if you need a specific week
print(f"League ID: {league_id}, Year: {year}, Week: {scoring_period_id}")

League ID: 64175, Year: 2026, Week: 5


In [3]:
# 1. Load mTeam Data
try:
    mTeam_data = load_view_json("mTeam", league_id, year, scoring_period_id)
    print("Successfully loaded mTeam_data")
except Exception as e:
    print(f"Error loading mTeam data: {e}")
    mTeam_data = {}

2026-05-03 09:11:05.741 
  command:

    streamlit run c:\Users\frogg\Anaconda3\envs\FantasySports_py3_12_3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-05-03 09:11:05.743 No runtime found, using MemoryCacheStorageManager


Error loading mTeam data: data\espn_json/mTeam_64175_2026_5.json not found. Please refresh data first.


In [4]:
# 2. Load Category Stats (This depends on mBoxscore data)
try:
    category_stats_df = get_category_stats(league_id, year, scoring_period_id)
    print(f"Successfully generated category_stats_df with shape: {category_stats_df.shape}")
except Exception as e:
    print(f"Error generating category stats: {e}")
    category_stats_df = pd.DataFrame()

2026-05-03 09:05:12.762 No runtime found, using MemoryCacheStorageManager
ERROR:data.espn_mlb_utilities:Error in get_category_stats for league_id=64175, year=2026, scoring_period_id=5
Traceback (most recent call last):
  File "c:\Users\frogg\Documents\python_scripts\fanatsy_sports\espn_fantasy\espn_flb_inseason\mlb_fantasy_dashboard_V2\data\espn_mlb_utilities.py", line 344, in get_category_stats
    data = load_view_json("mBoxscore", league_id, year, scoring_period_id)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\frogg\Anaconda3\envs\FantasySports_py3_12_3\Lib\site-packages\streamlit\runtime\caching\cache_utils.py", line 168, in wrapper
    return cached_func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\frogg\Anaconda3\envs\FantasySports_py3_12_3\Lib\site-packages\streamlit\runtime\caching\cache_utils.py", line 197, in __call__
    return self._get_or_create_cached_value(args, kwargs)
           ^^^^^^^^^^^^^^^

Successfully generated category_stats_df with shape: (0, 0)


### Inspect Category Stats
Check if the stats for week 1 are present and causing the imbalance.

In [5]:
if not category_stats_df.empty:
    display(category_stats_df.head())
    
    # Show unique weeks present in the data
    if 'matchupPeriodId' in category_stats_df.columns:
        print("Weeks in data:", category_stats_df['matchupPeriodId'].unique())
        
    # Inspect the teams that played in week 1 vs week 2
    if 'matchupPeriodId' in category_stats_df.columns and 'Team Names' in category_stats_df.columns:
        teams_w1 = category_stats_df[category_stats_df['matchupPeriodId'] == 1]['Team Names'].unique()
        teams_w2 = category_stats_df[category_stats_df['matchupPeriodId'] == 2]['Team Names'].unique()
        print(f"\nTeams with stats in Week 1 ({len(teams_w1)}): {teams_w1}")
        print(f"Teams with stats in Week 2 ({len(teams_w2)}): {teams_w2}")

### Call `get_standings_table`
Run the function to see the current output and the imbalance.

In [6]:
standings_df = get_standings_table(mTeam_data, category_stats_df)
display(standings_df)

""


### Debugging the Imbalance
If Week 1 only had 6 teams play, but the season started in Week 2, you may want to filter out `matchupPeriodId == 1` before it is passed to `get_standings_table` or directly inside `get_category_stats`.

In [8]:
# Try filtering out week 1
if not category_stats_df.empty and 'matchupPeriodId' in category_stats_df.columns:
    filtered_category_stats = category_stats_df[category_stats_df['matchupPeriodId'] >= 2]
    
    # Re-run standings table with filtered data
    filtered_standings_df = get_standings_table(mTeam_data, filtered_category_stats)
    display(filtered_standings_df)

,Rank,Team Names,Wins,Losses,Ties,GB,R,HR,RBI,SB,...,W,ERA,WHIP,SVHD,H,AB,ER,IP,BB,HA
0,1,Corb on the Cob,0,0,0,0.0,27.0,6.0,20.0,10.0,...,1.0,4.790323,1.693548,4.0,35,197,11,20.7,12,23
1,2,Team Better Than Matt,0,0,0,0.0,31.0,9.0,29.0,7.0,...,2.0,3.933775,1.092715,0.0,50,192,22,50.3,10,45
2,3,GoutFinger Fastball,0,0,0,0.0,25.0,2.0,23.0,7.0,...,4.0,4.939024,1.829268,3.0,52,205,15,27.3,18,32
3,4,Fear and Ignorance,0,0,0,0.0,27.0,4.0,20.0,6.0,...,1.0,7.090909,1.545455,3.0,51,214,26,33.0,18,33
4,5,Chicks dig the long ball,0,0,0,0.0,24.0,8.0,26.0,5.0,...,3.0,3.098361,1.303279,3.0,52,177,14,40.7,17,36
5,6,VOTE BRANDON- '24,0,0,0,0.0,19.0,5.0,14.0,6.0,...,1.0,3.634615,1.384615,6.0,35,188,14,34.7,19,29
6,7,Wade Boggs Carpet World,0,0,0,0.0,29.0,6.0,28.0,6.0,...,2.0,4.371429,1.342857,0.0,51,185,17,35.0,11,36
7,8,Oh No! We Suck Again!,0,0,0,0.0,29.0,4.0,22.0,6.0,...,0.0,3.919355,1.451613,2.0,41,185,9,20.7,5,25
8,9,Tanking for ever,0,0,0,0.0,16.0,4.0,28.0,5.0,...,3.0,3.715596,1.266055,5.0,52,201,15,36.3,14,32
9,10,Hard Body Karate,0,0,0,0.0,28.0,10.0,35.0,6.0,...,3.0,2.877049,0.836066,1.0,47,194,13,40.7,9,25
